# 07 — Data Preparation for All Holdout Experiments

Prepares h5ad datasets for 4 holdout groups. Group A (CD8-only) is already done
from notebook 05. This notebook creates datasets for Groups B, C, D.

| Group | Holdout | Also excluded | Description |
|-------|---------|---------------|-------------|
| A | CD8 (CL:0000625) | — | Already done (notebook 05) |
| B | CD8 (CL:0000625) | thymocytes (CL:0000893) | Cleaner CD8 holdout |
| C | CD4+CD8+thymocytes | — | All T cell subtypes held out |
| D | CD4 (CL:0000624) | — | Different cell type |

Each group produces 3 files:
1. `ae_training_<group>.h5ad` — scGen training data (all cells minus excluded)
2. `<group>_holdout.h5ad` — CellOT/IMPACT data (condition=species)
3. `<group>_holdout_swapped.h5ad` — CellOT data (condition=cell_type_status)

In [ ]:
import sys
sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")

import importlib
if "speciesot_helpers" in sys.modules:
    importlib.reload(sys.modules["speciesot_helpers"])

import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse as sp_sparse
from speciesot_helpers import (
    top_n_organisms_from_species,
    match_cells_by_celltype_tissue,
    align_adatas_biomart_one2one,
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print("Imports OK")

[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.


/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1. Load and Align Full Datasets (same as notebook 05)

In [2]:
human_dir = '/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_sapiens/'
mouse_dir = '/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_muris/'

human_adatas = top_n_organisms_from_species(human_dir, -1, 'human')
mouse_adatas = top_n_organisms_from_species(mouse_dir, -1, 'mouse')

human_combined = ad.concat(human_adatas, join="outer")
mouse_combined = ad.concat(mouse_adatas, join="outer")

print(f"Human combined: {human_combined.shape}")
print(f"Mouse combined: {mouse_combined.shape}")
print(f"Total cells:    {human_combined.n_obs + mouse_combined.n_obs}")

Human combined: (58852, 61759)
Mouse combined: (47802, 18024)
Total cells:    106654


In [3]:
print("Aligning genes via BioMart one-to-one orthologs ...")
mouse_all_aligned, human_all_aligned, ortholog_table = align_adatas_biomart_one2one(
    mouse_combined, human_combined
)

print(f"Ortholog pairs: {len(ortholog_table)}")
print(f"Mouse aligned: {mouse_all_aligned.shape}")
print(f"Human aligned: {human_all_aligned.shape}")

Aligning genes via BioMart one-to-one orthologs ...
Ortholog pairs: 14451
Mouse aligned: (47802, 14451)
Human aligned: (58852, 14451)


In [4]:
N_HVG = 1000

mouse_all_aligned.obs['condition'] = 'mouse'
human_all_aligned.obs['condition'] = 'human'

all_cells = ad.concat([mouse_all_aligned, human_all_aligned], join='inner')
print(f"Concatenated all cells: {all_cells.shape}")

sc.pp.highly_variable_genes(all_cells, n_top_genes=N_HVG, flavor='seurat')
hvg_genes = all_cells.var_names[all_cells.var.highly_variable].tolist()
print(f"Selected {len(hvg_genes)} HVGs")

mouse_all_hvg = mouse_all_aligned[:, hvg_genes].copy()
human_all_hvg = human_all_aligned[:, hvg_genes].copy()
mouse_all_hvg.obs['condition'] = 'mouse'
human_all_hvg.obs['condition'] = 'human'

print(f"Mouse HVG: {mouse_all_hvg.shape}")
print(f"Human HVG: {human_all_hvg.shape}")

Concatenated all cells: (106654, 14451)
Selected 1000 HVGs
Mouse HVG: (47802, 1000)
Human HVG: (58852, 1000)


In [5]:
mouse_matched_hvg, human_matched_hvg = match_cells_by_celltype_tissue(
    mouse_all_hvg, human_all_hvg
)

print(f"Matched mouse: {mouse_matched_hvg.shape}")
print(f"Matched human: {human_matched_hvg.shape}")

Matched mouse: (6418, 1000)
Matched human: (6418, 1000)


## 2. Define Holdout Groups and Utility Functions

In [6]:
BASE_DIR = '/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT'
DATASET_DIR = os.path.join(BASE_DIR, 'cellot/cellot_gpu/datasets/speciesot-human-mouse')
CT_COL = 'cell_type_ontology_term_id'

keep_obs = ['condition', 'species', 'cell_type_ontology_term_id', 'cell_type',
            'tissue_ontology_term_id', 'tissue', 'donor_id']

def clean_adata(adata):
    """Strip layers/obsm/uns and densify X for compatibility with older anndata in CellOT env."""
    obs_cols = [c for c in keep_obs if c in adata.obs.columns]
    X = adata.X
    if sp_sparse.issparse(X):
        X = np.array(X.todense())
    elif not isinstance(X, np.ndarray):
        X = np.array(X)
    return ad.AnnData(
        X=X.astype(np.float32),
        obs=adata.obs[obs_cols].copy(),
        var=pd.DataFrame(index=adata.var_names),
    )


GROUPS = {
    'B': {
        'name': 'cd8_nothymo',
        'description': 'CD8 holdout, thymocytes also excluded from training',
        'holdout_ids': ['CL:0000625'],
        'exclude_from_ae': ['CL:0000625', 'CL:0000893'],
        'holdout_label': 'cd8',
    },
    'C': {
        'name': 'tcell_subtypes',
        'description': 'All T cell subtypes held out (CD4+CD8+thymocytes)',
        'holdout_ids': ['CL:0000624', 'CL:0000625', 'CL:0000893'],
        'exclude_from_ae': ['CL:0000624', 'CL:0000625', 'CL:0000893'],
        'holdout_label': 'tcell_subtype',
    },
    'D': {
        'name': 'cd4',
        'description': 'CD4 holdout only',
        'holdout_ids': ['CL:0000624'],
        'exclude_from_ae': ['CL:0000624'],
        'holdout_label': 'cd4',
    },
}

print("Holdout groups defined:")
for gid, g in GROUPS.items():
    print(f"  Group {gid} ({g['name']}): {g['description']}")
    print(f"    holdout_ids:      {g['holdout_ids']}")
    print(f"    exclude_from_ae:  {g['exclude_from_ae']}")

Holdout groups defined:
  Group B (cd8_nothymo): CD8 holdout, thymocytes also excluded from training
    holdout_ids:      ['CL:0000625']
    exclude_from_ae:  ['CL:0000625', 'CL:0000893']
  Group C (tcell_subtypes): All T cell subtypes held out (CD4+CD8+thymocytes)
    holdout_ids:      ['CL:0000624', 'CL:0000625', 'CL:0000893']
    exclude_from_ae:  ['CL:0000624', 'CL:0000625', 'CL:0000893']
  Group D (cd4): CD4 holdout only
    holdout_ids:      ['CL:0000624']
    exclude_from_ae:  ['CL:0000624']


## 3. Inspect Cell Counts for Each Group

In [7]:
all_hvg = ad.concat([mouse_all_hvg, human_all_hvg], join='inner')
matched_hvg = ad.concat([mouse_matched_hvg, human_matched_hvg], join='inner')

T_CELL_IDS = {
    'CL:0000084': 'T cell',
    'CL:0000893': 'thymocyte',
    'CL:0000624': 'CD4+ T cell',
    'CL:0000625': 'CD8+ T cell',
}

print("T cell family in matched dataset:")
print("=" * 70)
for cid, name in T_CELL_IDS.items():
    mask = matched_hvg.obs[CT_COL].astype(str) == cid
    n_total = mask.sum()
    n_mouse = (mask & (matched_hvg.obs['condition'] == 'mouse')).sum()
    n_human = (mask & (matched_hvg.obs['condition'] == 'human')).sum()
    print(f"  {cid} ({name}): {n_total} total ({n_mouse} mouse, {n_human} human)")

print(f"\nTotal matched cells: {matched_hvg.n_obs}")
print(f"Total all cells: {all_hvg.n_obs}")

for gid, g in GROUPS.items():
    excluded = set(g['exclude_from_ae'])
    ae_mask = ~all_hvg.obs[CT_COL].astype(str).isin(excluded)
    holdout_mask = matched_hvg.obs[CT_COL].astype(str).isin(g['holdout_ids'])
    print(f"\nGroup {gid} ({g['name']}):")
    print(f"  AE training cells: {ae_mask.sum()} (excluded {(~ae_mask).sum()})")
    print(f"  Holdout cells in matched data: {holdout_mask.sum()}")

T cell family in matched dataset:
  CL:0000084 (T cell): 204 total (102 mouse, 102 human)
  CL:0000893 (thymocyte): 910 total (455 mouse, 455 human)
  CL:0000624 (CD4+ T cell): 190 total (95 mouse, 95 human)
  CL:0000625 (CD8+ T cell): 390 total (195 mouse, 195 human)

Total matched cells: 12836
Total all cells: 106654

Group B (cd8_nothymo):
  AE training cells: 103201 (excluded 3453)
  Holdout cells in matched data: 390

Group C (tcell_subtypes):
  AE training cells: 101202 (excluded 5452)
  Holdout cells in matched data: 1490

Group D (cd4):
  AE training cells: 104655 (excluded 1999)
  Holdout cells in matched data: 190


## 4. Generate Datasets for Each Group

In [8]:
for gid, g in GROUPS.items():
    group_name = g['name']
    holdout_ids = set(g['holdout_ids'])
    exclude_ids = set(g['exclude_from_ae'])
    label = g['holdout_label']

    print(f"\n{'=' * 70}")
    print(f"GROUP {gid}: {g['description']}")
    print(f"{'=' * 70}")

    # --- File 1: AE training data (excluded cells removed) ---
    ae_all = ad.concat([mouse_all_hvg, human_all_hvg], join='inner')
    ae_mask = ~ae_all.obs[CT_COL].astype(str).isin(exclude_ids)
    ae_data = clean_adata(ae_all[ae_mask].copy())

    ae_path = os.path.join(DATASET_DIR, f'ae_training_{group_name}.h5ad')
    ae_data.write_h5ad(ae_path)
    print(f"  AE training: {ae_data.n_obs} cells -> {ae_path}")
    print(f"    Conditions: {dict(ae_data.obs['condition'].value_counts())}")

    # --- File 2: CellOT unswapped (condition=species, for IMPACT framing) ---
    mouse_m = mouse_matched_hvg.copy()
    human_m = human_matched_hvg.copy()
    mouse_m.obs['condition'] = 'mouse'
    human_m.obs['condition'] = 'human'
    cellot_data = ad.concat([mouse_m, human_m], join='inner')
    cellot_data = clean_adata(cellot_data)

    cellot_path = os.path.join(DATASET_DIR, f'{group_name}_holdout.h5ad')
    cellot_data.write_h5ad(cellot_path)
    n_holdout = cellot_data.obs[CT_COL].astype(str).isin(holdout_ids).sum()
    print(f"  CellOT (unswapped): {cellot_data.n_obs} cells, {n_holdout} holdout -> {cellot_path}")

    # --- File 3: CellOT swapped (condition=cell_type_status, for CellOT framing) ---
    mouse_s = mouse_matched_hvg.copy()
    human_s = human_matched_hvg.copy()
    swapped = ad.concat([mouse_s, human_s], join='inner')
    swapped.obs['species'] = swapped.obs['condition'].values
    is_holdout = swapped.obs[CT_COL].astype(str).isin(holdout_ids)
    swapped.obs['condition'] = np.where(is_holdout, label, f'non_{label}')
    swapped = clean_adata(swapped)

    swapped_path = os.path.join(DATASET_DIR, f'{group_name}_holdout_swapped.h5ad')
    swapped.write_h5ad(swapped_path)
    print(f"  CellOT (swapped): {swapped.n_obs} cells -> {swapped_path}")
    print(f"    condition values: {dict(swapped.obs['condition'].value_counts())}")
    print(f"    species values:   {dict(swapped.obs['species'].value_counts())}")

print(f"\n{'=' * 70}")
print("All datasets generated.")


GROUP B: CD8 holdout, thymocytes also excluded from training
  AE training: 103201 cells -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/ae_training_cd8_nothymo.h5ad
    Conditions: {'human': np.int64(57399), 'mouse': np.int64(45802)}
  CellOT (unswapped): 12836 cells, 390 holdout -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_nothymo_holdout.h5ad
  CellOT (swapped): 12836 cells -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_nothymo_holdout_swapped.h5ad
    condition values: {'non_cd8': np.int64(12446), 'cd8': np.int64(390)}
    species values:   {'human': np.int64(6418), 'mouse': np.int64(6418)}

GROUP C: All T cell subtypes held out (CD4+CD8+thymocytes)
  AE training: 101202 cells -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/ae_training_tcell_subtypes.h5ad
    Condition

## 5. Create v07-compatible Copies

The CellOT conda environment uses an older anndata that needs v0.7 format.
The `clean_adata()` function already strips incompatible fields, so the files
written above should be compatible. We save explicit `_v07` copies to match
the existing naming convention used by the task configs.

In [9]:
import shutil

for gid, g in GROUPS.items():
    group_name = g['name']
    for suffix in ['', '_swapped']:
        # AE training
        ae_src = os.path.join(DATASET_DIR, f'ae_training_{group_name}.h5ad')
        ae_dst = os.path.join(DATASET_DIR, f'ae_training_{group_name}_v07.h5ad')
        if not os.path.exists(ae_dst):
            shutil.copy2(ae_src, ae_dst)
            print(f"Copied {ae_src} -> {ae_dst}")

    # CellOT holdout
    for suffix in ['', '_swapped']:
        src = os.path.join(DATASET_DIR, f'{group_name}_holdout{suffix}.h5ad')
        dst = os.path.join(DATASET_DIR, f'{group_name}_holdout{suffix}_v07.h5ad')
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            print(f"Copied {src} -> {dst}")

print("\nv07 copies created.")

Copied /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/ae_training_cd8_nothymo.h5ad -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/ae_training_cd8_nothymo_v07.h5ad
Copied /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_nothymo_holdout.h5ad -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_nothymo_holdout_v07.h5ad
Copied /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_nothymo_holdout_swapped.h5ad -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_nothymo_holdout_swapped_v07.h5ad
Copied /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/ae_training_tcell_subtypes.h5ad -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datas

## 6. Summary

In [10]:
print("Files created:")
print("=" * 70)
for f in sorted(os.listdir(DATASET_DIR)):
    if f.endswith('.h5ad'):
        fpath = os.path.join(DATASET_DIR, f)
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {f:55s} {size_mb:6.1f} MB")

print(f"\nNext steps:")
print(f"  1. Create task YAML configs for each group (see configs/tasks/)")
print(f"  2. Submit sbatch jobs: scGen first, then IMPACT + CellOT")
print(f"  3. Run evaluation notebook (08) after all training completes")

Files created:
  ae_training_cd4.h5ad                                     407.3 MB
  ae_training_cd4_v07.h5ad                                 407.3 MB
  ae_training_cd8_nothymo.h5ad                             401.7 MB
  ae_training_cd8_nothymo_v07.h5ad                         401.7 MB
  ae_training_expanded.h5ad                                407.3 MB
  ae_training_expanded_compat.h5ad                         415.1 MB
  ae_training_expanded_v07.h5ad                            407.3 MB
  ae_training_tcell_subtypes.h5ad                          393.9 MB
  ae_training_tcell_subtypes_v07.h5ad                      393.9 MB
  cd4_holdout.h5ad                                          50.0 MB
  cd4_holdout_swapped.h5ad                                  50.0 MB
  cd4_holdout_swapped_v07.h5ad                              50.0 MB
  cd4_holdout_v07.h5ad                                      50.0 MB
  cd8_holdout.h5ad                                          50.0 MB
  cd8_holdout_compat.h5ad        